# Delivering Outputs to Object Storage

## What you'll learn

- Send a Modal-executed tool op's outputs to your own S3-compatible
  bucket with `output_store` — one config field, no redeploy
- Follow the delivery: the worker uploads a tarball under your prefix
  and the client fetches it through a presigned URL
- Verify and clean up the delivered objects

**Prerequisites:** [Running on Modal](04-modal-execution.ipynb).
**Estimated time:** 10 minutes.
**GPU required:** No


:::{note}
This tutorial requires Modal credentials, a deployed `wait_tool`
endpoint whose deployment carries an object-store secret (setup below),
and an S3-compatible bucket. The walkthrough uses Cloudflare R2; AWS S3
and MinIO work identically. Code cells are shown for reference and are
not executed in the docs build.
:::


## Why a store

By default a Modal-executed op returns its outputs inline, bounded at
100 MB. Naming an `output_store` removes the bound: the worker delivers
the output tarball straight to your bucket and only a pointer rides the
result. The destination is **request data** — two callers of one
deployed endpoint can deliver to two different buckets, per run. The
full contract (credentials, lifecycle, external consumers) is in
[Object-store output delivery](../../how-to-guides/configuring-execution.md#object-store-output-delivery).


## One-time setup

The worker uploads with its **own** credentials, injected as a Modal
Secret. Create the secret once, name it in the op's config, and deploy:

```bash
modal secret create r2-artisan \
  AWS_ACCESS_KEY_ID=... AWS_SECRET_ACCESS_KEY=... \
  AWS_REGION=auto AWS_ENDPOINT_URL=https://<account>.r2.cloudflarestorage.com
```

```python
class MyTool(OperationDefinition):
    ...
    compute_provider = ComputeProvider(
        modal=ModalComputeConfig(secrets=["r2-artisan"])
    )
```

```bash
artisan modal deploy my_tool
```

For the `wait_tool` example endpoint used here, the repo's live
integration test deploys it with the secret attached:
`pixi run -e dev test-modal-endpoint`.

This notebook also reads your bucket coordinates from the environment
(or the repo-root `.env`):

```bash
# .env
AWS_ACCESS_KEY_ID=...            # used below to verify the delivery
AWS_SECRET_ACCESS_KEY=...
ARTISAN_S3_ENDPOINT_URL=https://<account>.r2.cloudflarestorage.com
ARTISAN_S3_BUCKET=my-bucket
```


In [1]:
from __future__ import annotations

from artisan.operations.examples import DataGenerator, WaitTool
from artisan.orchestration import PipelineManager
from artisan.schemas.operation_config.compute import (
    ComputeProvider,
    ModalComputeConfig,
)
from artisan.utils import tutorial_setup
from artisan.utils.env_file import env_or_dotenv

env = tutorial_setup("modal_r2_outputs", clean=True)

In [2]:
import os
import uuid

BUCKET = env_or_dotenv("ARTISAN_S3_BUCKET")
ENDPOINT = env_or_dotenv("ARTISAN_S3_ENDPOINT_URL")
assert BUCKET, "set ARTISAN_S3_BUCKET (see setup above)"
assert ENDPOINT, "set ARTISAN_S3_ENDPOINT_URL (see setup above)"

# ambient credentials for the verification cells at the end — the same
# variable shapes the worker's Modal Secret injects on the other side
for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    os.environ.setdefault(key, env_or_dotenv(key) or "")
os.environ.setdefault("AWS_ENDPOINT_URL", ENDPOINT)

OUTPUT_STORE = f"s3://{BUCKET}/artisan-tutorial/{uuid.uuid4().hex[:8]}"
print(f"outputs will be delivered under {OUTPUT_STORE}")

outputs will be delivered under s3://artisan-test/artisan-tutorial/1d01f205


## Run on Modal, deliver to the bucket

`output_store` joins the modal config like any other field. Here it
rides a per-step override — the deployment is untouched, and a step
without the override still returns outputs inline:


In [3]:
pipeline = PipelineManager.create(
    name="modal_r2_outputs",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

gen = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 2, "seed": 42},
)

pipeline.run(
    operation=WaitTool,
    name="wait",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
    compute_provider=ComputeProvider(
        active="modal",
        modal=ModalComputeConfig(output_store=OUTPUT_STORE),
    ),
)

result = pipeline.finalize()
assert result["overall_success"]
print("pipeline complete")

09:16:42.978 | INFO    | artisan.orchestration.pipeline_manager - Pipeline 'modal_r2_outputs' initialized (run_id=modal_r2_outputs_20260612_161642_92581fd5)

09:16:42.979 | INFO    | artisan.orchestration.pipeline_manager -   delta_root: /Users/ach/git/artisan/dev/docs/tutorials/07-compute-backends/runs/modal_r2_outputs/delta

09:16:42.980 | INFO    | artisan.orchestration.pipeline_manager -   staging_root: /Users/ach/git/artisan/dev/docs/tutorials/07-compute-backends/runs/modal_r2_outputs/staging

09:16:42.985 | INFO    | artisan.orchestration.pipeline_manager - Step 0 (generate) starting... [step_runner=local]

09:16:44.414 | INFO    | artisan.orchestration.engine.dispatch - Collected results from 1 futures

...

09:16:58.976 | INFO    | artisan.orchestration.pipeline_manager - Pipeline 'modal_r2_outputs' complete: 2 steps, all succeeded

09:16:58.976 | INFO    | artisan.orchestration.pipeline_manager -   Step 0: generate         1.7s  [1/1]

09:16:58.977 | INFO    | artisan.orchestration.pipeline_manager -   Step 1: wait             14.3s  [2/2]

09:16:58.977 | INFO    | artisan.orchestration.pipeline_manager -   Total: 16.0s

pipeline complete


## What happened

For each artifact, the worker ran the tool, tarred its outputs, and
uploaded `<output_store>/<op-name>/<uuid>.tar.gz` with the deployment's
secret. The result carried no bulk bytes — only the tarball's URI plus
a presigned download URL (valid 7 days, matching how long Modal retains
the result). The pipeline client fetched through that URL and committed
artifacts exactly as an inline run would — the lifecycle downstream is
identical.

The same pointer serves consumers outside artisan: anyone polling the
endpoint gets the presigned URL and fetches results with one plain HTTP
GET, no storage credentials. Callers whose buckets the worker cannot
reach send a presigned PUT URL instead — see
[capability mode](../../how-to-guides/configuring-execution.md#presigned-puts-and-external-consumers-capability-mode).


## Verify the delivery

Two artifacts fanned out as two endpoint calls, so two tarballs landed
under the prefix. List them, and build a link to browse them in your
provider's web console:


In [4]:
from urllib.parse import urlparse

import s3fs

fs = s3fs.S3FileSystem()
for key in fs.find(OUTPUT_STORE.removeprefix("s3://")):
    print(key)

host = urlparse(ENDPOINT).netloc
prefix = OUTPUT_STORE.removeprefix(f"s3://{BUCKET}/")
if host.endswith(".r2.cloudflarestorage.com"):
    account_id = host.split(".", 1)[0]
    console = f"https://dash.cloudflare.com/{account_id}/r2/default/buckets/{BUCKET}"
elif host.endswith(".amazonaws.com"):
    console = f"https://s3.console.aws.amazon.com/s3/buckets/{BUCKET}?prefix={prefix}/"
else:
    console = None  # self-hosted stores (MinIO, ...) serve their own console
print(
    "\nbrowse the delivered files:", console or f"open your store's console for {host}"
)

artisan-test/artisan-tutorial/1d01f205/wait_tool/00c47ae7b959477ab41680f2824fb118.tar.gz
artisan-test/artisan-tutorial/1d01f205/wait_tool/4d4a4ae464a54707a6f64ffcf6a9183d.tar.gz

browse the delivered files: https://dash.cloudflare.com/<account-id>/r2/default/buckets/artisan-test


## Clean up

Artisan never deletes delivered tarballs — pair real destination
prefixes with a bucket lifecycle policy. For the tutorial prefix, remove
the objects directly:


In [5]:
fs.rm(OUTPUT_STORE.removeprefix("s3://"), recursive=True)
print("removed", OUTPUT_STORE)

removed s3://artisan-test/artisan-tutorial/1d01f205


## Summary

- `output_store` on the modal config delivers a Modal-executed op's
  outputs to any S3-compatible bucket, with no size bound and no
  redeploy — the destination travels with each request
- The worker uploads with its Modal Secret's credentials; consumers
  fetch through a presigned URL, credential-free
- Without `output_store`, behavior is unchanged: outputs return inline,
  bounded at 100 MB

## Next steps

- [Object-store output delivery](../../how-to-guides/configuring-execution.md#object-store-output-delivery)
  — the full contract: IAM scoping, lifecycle, external consumers
- [Configure S3-Compatible Storage](../../how-to-guides/configuring-s3.md)
  — putting the pipeline's own storage (Delta tables, staging, files)
  on a bucket
- [Running on Modal](04-modal-execution.ipynb) — the endpoint model
  this builds on
